In [2]:
pip install pinecone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\steven\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
pip install openpyxl

   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ---- ----------------------------------- 30.7/250.9 kB 1.4 MB/s eta 0:00:01
   --------------------------- ------------ 174.1/250.9 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 250.9/250.9 kB 3.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\steven\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
import os
import pandas as pd
file_path = "Synethic LDI restrictions.xlsx"

In [20]:
df = pd.read_excel(file_path)

In [21]:
df

,Rule ID,Mandate transcript
0,87897,The cumulative PV01 of the benchmark at each t...
1,43203,The duration of liabilities should match withi...
2,22329,The Bond PV01 of the portfolio shall not devia...
3,5507,The cumulative PV01 of the benchmark at each t...
4,51691,Asset allocation should not deviate by more th...
...,...,...
176,45066,The duration of liabilities should match withi...
177,78149,Asset allocation should not deviate by more th...
178,24461,The PV01 of the portfolio shall not deviate by...
179,1763,The PV01 of the portfolio shall not deviate by...


In [23]:
# select the embedding fields to create the lookup key for the semantic search

In [41]:
from pinecone import ServerlessSpec
import openai
from openai import OpenAI
openai_key = os.getenv("OPENAI_API_KEY")
from pinecone import Pinecone

In [42]:
#initialise an openai client
client = OpenAI(api_key = openai_key)
client

In [43]:
pc = Pinecone(api_key= os.getenv("PINECONE_API_KEY"))
pc

In [44]:
# created an index
index = pc.Index("guideline-bot")

In [60]:
#call open ai to embedd your data
def get_embedding(text_to_embed):
    response = client.embeddings.create(
        model = "text-embedding-3-small",
        input = [text_to_embed]
    )
    embedding = response.data[0].embedding
    return embedding

In [61]:
df['embedding'] = df["Mandate transcript"].astype(str).apply(get_embedding)

In [86]:
def upsert_to_pine(df,index):
        
    for idx,row in df.iterrows():
        rule_id = str(row['Rule ID'])
        mandate_transcript = row['Mandate transcript']
        embedding = row['embedding']

        metadata = {"mandate_transcript": mandate_transcript}

        #upsert into pinecone
        index.upsert([(rule_id,embedding,metadata)])


In [88]:
#upsert to pine
upsert_to_pine(df,index)

Recap:
1. we had to pd.read our excel into a dataframe
2. We need to ensure that empty rows are removed or columns as this can confuse the AI#
3. We need to initialise pinecone and openai keys
4. Always call the 'client' to reach an endpoint on openai 
5. You need to call the embedding model from OpenAI create vectors for the mandate transcript to be able to do semantic searches. 
6. Rule IDs = will be the unique identifiers
7. Metadata = is the mandate transcript which is what the bot will look up and query in the vector DB.
8. The embeddings will help it choose the mandate transcripts which have the closest similarity.
9. We need to vectorize our query or maybe the whole compare document!

In [97]:
print(index.describe_index_stats())

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 181}},
 'total_vector_count': 181}
